# Broodpile Pipeline

This notebook runs the broodpile workflow end to end using the existing repo scripts.
The configuration cell below is an example only and uses placeholder paths that must be edited before execution.

Steps:
1. Run YOLO segmentation and write raw `*.jsonl.gz` detections.
2. Detect colony masks and write `colonies.pkl`.
3. Filter raw YOLO JSONL detections into `*.filtered.jsonl.gz`.
4. Combine the filtered detections with colony masks and write per-colony CSVs.

The notebook uses strict inline constants for `width`, `height`, and `fps`, and processes only the `larvae` class downstream.

## Inputs
- `project_root`: example root that holds model weights, videos, the colony mask, and outputs.
- `runs`: example run definitions built from `run_names`; edit them to match your dataset layout.
- `model_path` and `image_path`: required inputs that must already exist before the notebook runs.
- `colonies_pkl`: output file written by colony-mask processing.
- `width`, `height`, and `fps`: experiment constants used downstream.
- `filter_cfg`: example filtering parameters passed directly to `yolo_filtering.py`.
- Each run has its own `prediction_dir`, `filtered_dir`, `output_dir`, and `video_dir`.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Example configuration for publication.
# Replace these paths and run names with your local dataset layout before running the notebook.
project_root = Path("/path/to/broodpile_example").expanduser()
video_root = project_root / "videos"
output_root = project_root / "outputs"
python = sys.executable

model_path = project_root / "models/best.pt"
image_path = project_root / "mask.png"
colonies_pkl = output_root / "colonies.pkl"

run_names = [
    "run_01",
    "run_02",
]

runs = []
for run_name in run_names:
    run_root = output_root / run_name
    runs.append(
        {
            "name": run_name,
            "video_input_path": video_root / run_name,
            "prediction_dir": run_root / "predictions_raw",
            "filtered_dir": run_root / "predictions_filtered",
            "output_dir": run_root / "combine",
            "video_dir": video_root / run_name,
        }
    )


orientation = "clockwise"
width = 4512
height = 4512
fps = 25.0
use_fixed_threshold = False
enable_preview = False
segmentation_jobs = 1
write_segmentation_overlay = False

# Filtering config passed directly to yolo_filtering.filter_all_files.
filter_cfg = {
    "paths": {
        "output_dir": "",
    },
    "iou_threshold": 0.3,
    "mask_buffer_px": 0.0,
    "conf_threshold":  0.1,
    "final_conf_threshold": 0.5,
    "class_names": ["larvae"],
    "classes_to_process": [0],
    "containment_fraction": 0.5,
    "parallel": {
        "enabled": True,
        "workers": 4,
    },
    "overlay_colors": [
        [34, 139, 230],
    ],
    "test": {
        "enabled": enable_preview,
        "video_dir": "",
        "preview_fps": fps,
        "edge_thickness": 3,
    },
}

required_paths = [
    model_path,
    image_path,
]
for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(path)
for run in runs:
    if not run["video_input_path"].exists():
        raise FileNotFoundError(run["video_input_path"])

output_root.mkdir(parents=True, exist_ok=True)
for run in runs:
    run["prediction_dir"].mkdir(parents=True, exist_ok=True)
    run["filtered_dir"].mkdir(parents=True, exist_ok=True)
    run["output_dir"].mkdir(parents=True, exist_ok=True)

## 1. Run segmentation

This calls `yolo_segment.sh` to generate raw `*.jsonl.gz` detections in `prediction_dir`.

In [ ]:
for run in runs:
    cmd = [
        "bash",
        str("yolo_segment.sh"),
        str(model_path),
        str(run["video_input_path"]),
        str(run["prediction_dir"]),
        str(segmentation_jobs),
    ]
    if write_segmentation_overlay:
        cmd.append(str(run["prediction_dir"] / "overlays"))

    subprocess.run(cmd, check=True)
    jsonl_files = sorted(run["prediction_dir"].glob("*.jsonl.gz"))
    if not jsonl_files:
        raise FileNotFoundError(f"No .jsonl.gz files were generated in {run['prediction_dir']}")
    print(f"[{run['name']}] Wrote {len(jsonl_files)} raw prediction files to {run['prediction_dir']}")


## 2. Build colony masks

This calls `process_colony_masks.py` directly. It writes `colonies.pkl` and the labeled mask overlay next to the source mask image.

In [ ]:
cmd = [
    python,
    str("process_colony_masks.py"),
    "--image-path", str(image_path),
    "--colonies-file", str(colonies_pkl),
    "--orientation", orientation,
    "--width", str(width),
    "--height", str(height),
]
if use_fixed_threshold:
    cmd.append("--fixed-threshold")

subprocess.run(cmd, check=True)
if not colonies_pkl.exists():
    raise FileNotFoundError(colonies_pkl)
print(f"Wrote {colonies_pkl}")


## 3. Filter detections

This imports `yolo_filtering.py` and runs `filter_one_file` directly so the notebook can pass a strict config dictionary without relying on a hard-coded config file path.

In [ ]:
from yolo_filtering import filter_all_files

for run in runs:
    filter_cfg["paths"]["output_dir"] = str(run["filtered_dir"])
    filter_cfg["test"]["video_dir"] = str(run["video_dir"])
    jsonl_files = sorted([p for p in run["prediction_dir"].iterdir() if p.name.endswith(".jsonl.gz")])
    if not jsonl_files:
        raise FileNotFoundError(f"No .jsonl.gz files found in {run['prediction_dir']}")

    filter_all_files(jsonl_files, run["filtered_dir"], filter_cfg)

    print(f"[{run['name']}] Filtered {len(jsonl_files)} files into {run['filtered_dir']}")


## 4. Combine colony metrics

This calls `combine_segmentation_colony.py` for each run and writes one CSV set per output directory.

In [8]:
run["filtered_dir"]

PosixPath('/media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264057/filter_predictions_filtend')

In [3]:
for run in runs:
    cmd = [
        python,
        str("combine_segmentation_colony.py"),
        str(run["filtered_dir"]),
        str(colonies_pkl),
        "-o", str(run["output_dir"]),
    ]

    subprocess.run(cmd, check=True)
    print(f"[{run['name']}] Wrote outputs to {run['output_dir']}")


[INFO] Loading colonies from /media/AntGate/user/patrick/shared/broodpile_pipeline_output/colonies.pkl
[INFO] Loaded 4 colonies
[INFO] Found 186 filtered JSONL files


Processing JSONL files:  99%|█████████▉| 185/186 [30:29<00:22, 22.92s/it]

[INFO] Writing output CSVs...


Processing JSONL files: 100%|██████████| 186/186 [30:40<00:00,  9.89s/it]


  Wrote 1621455 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/combine_final_filtend/colony_01_detections.csv
  Wrote 1613617 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/combine_final_filtend/colony_02_detections.csv
  Wrote 1617317 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/combine_final_filtend/colony_03_detections.csv
  Wrote 1618772 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/combine_final_filtend/colony_04_detections.csv
[INFO] Done. Output CSVs in /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264048/combine_final_filtend
[patrick_oct17_alexa647_20241017_162401.40264048] Wrote outputs to /media/AntGate/user/patrick/shared/broodpi

...

[INFO] Writing output CSVs...
  Wrote 1597859 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264049/combine_final_filtend/colony_01_detections.csv
  Wrote 1592990 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264049/combine_final_filtend/colony_02_detections.csv
  Wrote 1596050 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264049/combine_final_filtend/colony_03_detections.csv
  Wrote 1589289 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264049/combine_final_filtend/colony_04_detections.csv
[INFO] Done. Output CSVs in /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264049/combine_final_filtend
[patrick_oct17_alexa647_20241017_162401.40264049] Wrote outputs to /media/AntGa

Processing JSONL files: 100%|██████████| 186/186 [28:30<00:00,  9.20s/it]


[INFO] Writing output CSVs...
  Wrote 1618143 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264052/combine_final_filtend/colony_01_detections.csv
  Wrote 1615089 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264052/combine_final_filtend/colony_02_detections.csv
  Wrote 1597088 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264052/combine_final_filtend/colony_03_detections.csv
  Wrote 1605869 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264052/combine_final_filtend/colony_04_detections.csv
[INFO] Done. Output CSVs in /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264052/combine_final_filtend
[patrick_oct17_alexa647_20241017_162401.40264052] Wrote outputs to /media/AntGa

Processing JSONL files: 100%|██████████| 188/188 [23:34<00:00,  7.53s/it]


[INFO] Writing output CSVs...
  Wrote 1653539 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264057/combine_final_filtend/colony_01_detections.csv
  Wrote 1664076 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264057/combine_final_filtend/colony_02_detections.csv
  Wrote 1653583 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264057/combine_final_filtend/colony_03_detections.csv
  Wrote 1616975 records to /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264057/combine_final_filtend/colony_04_detections.csv
[INFO] Done. Output CSVs in /media/AntGate/user/patrick/shared/broodpile_pipeline_output/patrick_oct17_alexa647_20241017_162401.40264057/combine_final_filtend
[patrick_oct17_alexa647_20241017_162401.40264057] Wrote outputs to /media/AntGa

## Outputs

The pipeline should now have:
- `colonies.pkl`
- raw `*.jsonl.gz` files in each run prediction directory
- `*.filtered.jsonl.gz` files in each run filtered directory
- `colony_XX_detections.csv` files in each run combine output directory

In [4]:
print("colonies.pkl:", colonies_pkl.exists())
for run in runs:
    print(f"[{run['name']}] raw jsonl files:", len(list(run["prediction_dir"].glob("*.jsonl.gz"))))
    print(f"[{run['name']}] filtered files:", len(list(run["filtered_dir"].glob("*.filtered.jsonl.gz"))))
    print(f"[{run['name']}] csv files:", len(list(run["output_dir"].glob("colony_*_detections.csv"))))


colonies.pkl: True
[patrick_oct17_alexa647_20241017_162401.40264048] raw jsonl files: 188
[patrick_oct17_alexa647_20241017_162401.40264048] filtered files: 186
[patrick_oct17_alexa647_20241017_162401.40264048] csv files: 4
[patrick_oct17_alexa647_20241017_162401.40264049] raw jsonl files: 188
[patrick_oct17_alexa647_20241017_162401.40264049] filtered files: 185
[patrick_oct17_alexa647_20241017_162401.40264049] csv files: 4
[patrick_oct17_alexa647_20241017_162401.40264052] raw jsonl files: 188
[patrick_oct17_alexa647_20241017_162401.40264052] filtered files: 186
[patrick_oct17_alexa647_20241017_162401.40264052] csv files: 4
[patrick_oct17_alexa647_20241017_162401.40264057] raw jsonl files: 188
[patrick_oct17_alexa647_20241017_162401.40264057] filtered files: 188
[patrick_oct17_alexa647_20241017_162401.40264057] csv files: 4
